In [34]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm 
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler


Note: LinearRegression/Lasso/Elastic Net/Ridge did not make sense for the fraud use case (binary classification). So I instead use Logistic Reg with different penalty

In [35]:
random_state=42

In [36]:
classification_reports = []
classification_report_keys = []

## Data

In [ ]:
# Import IBM dataset
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ibm_hi_small_trans_cleaned.csv')
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,3697.340000,0,3697.340000,3697.340000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,0.010000,0,0.010000,0.010000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,14675.570000,0,14675.570000,14675.570000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,2806.970000,0,2806.970000,2806.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,36682.970000,0,36682.970000,36682.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,0.154978,0,3107.386389,3107.386389,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,0.108128,0,2168.020464,2168.020464,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,0.004988,0,100.011894,100.011894,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,0.038417,0,770.280058,770.280058,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


In [38]:
df.drop(columns=['Timestamp', 'From Bank', 'To Bank', 'Account', 'Account.1'], inplace=True)

## Undersampling

In [39]:
from collections import Counter


X = df.drop(columns='Is Laundering')
y = df['Is Laundering']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 5073168, 1: 5177})
Resampled dataset shape: Counter({0: 5177, 1: 5177})


## Split Data

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

## Baseline Models
#### (No parameter optimization, feature scaling, or cross validation)

### Linear Regression

In [41]:
model = LogisticRegression()

In [42]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [43]:
y_pred = model.predict(X_test)

In [44]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Logistic Regression')

              precision    recall  f1-score   support

           0       1.00      0.01      0.02      1036
           1       0.50      1.00      0.67      1035

    accuracy                           0.50      2071
   macro avg       0.75      0.50      0.34      2071
weighted avg       0.75      0.50      0.34      2071



### Ridge

In [45]:
from sklearn.linear_model import RidgeClassifier


ridge_model = RidgeClassifier(random_state=random_state)

In [46]:
ridge_model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.80134e-23): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,class_weight,None
,solver,'auto'
,positive,False
,random_state,42


In [47]:
y_pred = ridge_model.predict(X_test)

In [48]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Ridge Classifier')


              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



### Lasso

In [49]:
lasso_model = LogisticRegression(penalty='l1', solver='liblinear')

In [50]:
lasso_model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,penalty,'l1'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [51]:
y_pred = lasso_model.predict(X_test)

In [52]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline L1 (Lasso) Logistic Regression')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



## Scaling Data

In [53]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Parameter Tuning

### Logistic Regression

In [54]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [55]:
# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(n_splits=5)

In [56]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_recall_optuna_results)

[I 2026-05-27 19:41:09,094] A new study created in memory with name: no-name-807d5d35-8f4b-4459-9a23-d61ac98b9281
[I 2026-05-27 19:41:11,071] Trial 0 finished with value: 0.8597323473365851 and parameters: {'C': 0.9220621775694158, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:41:12,319] Trial 1 finished with value: 0.8597323473365851 and parameters: {'C': 0.8497322323162669, 'Class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:41:13,782] Trial 2 finished with value: 0.8597323473365851 and parameters: {'C': 0.5550396966626763, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:41:15,213] Trial 3 finished with value: 0.8597323473365851 and parameters: {'C': 0.5037153933289643, 'Class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:41:15,379] Trial 4 finished with value: 0.8597323473365851 and parameters: {'C': 0.5771305715701048, 

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
92,92,0.861423,2026-05-27 19:41:29.427533,2026-05-27 19:41:29.458626,0 days 00:00:00.031093,0.003649,None,COMPLETE
94,94,0.861423,2026-05-27 19:41:29.610310,2026-05-27 19:41:29.743848,0 days 00:00:00.133538,0.001062,None,COMPLETE
99,99,0.860939,2026-05-27 19:41:30.360154,2026-05-27 19:41:30.500908,0 days 00:00:00.140754,0.004529,balanced,COMPLETE
0,0,0.859732,2026-05-27 19:41:09.095404,2026-05-27 19:41:11.071118,0 days 00:00:01.975714,0.922062,balanced,COMPLETE
4,4,0.859732,2026-05-27 19:41:15.214762,2026-05-27 19:41:15.379290,0 days 00:00:00.164528,0.577131,None,COMPLETE


In [57]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_f1_optuna_results)

[I 2026-05-27 19:41:30,521] A new study created in memory with name: no-name-0c26239a-c4b8-4393-8a04-0f228f127645
[I 2026-05-27 19:41:30,694] Trial 0 finished with value: 0.8770744551069047 and parameters: {'C': 0.8267229218226069, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8770744551069047.
[I 2026-05-27 19:41:30,852] Trial 1 finished with value: 0.8771824847322993 and parameters: {'C': 0.40594143396134397, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.8771824847322993.
[I 2026-05-27 19:41:31,024] Trial 2 finished with value: 0.8771824847322993 and parameters: {'C': 0.7176124122418197, 'Class_weight': None}. Best is trial 1 with value: 0.8771824847322993.
[I 2026-05-27 19:41:31,177] Trial 3 finished with value: 0.8770744551069047 and parameters: {'C': 0.7805943006783158, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.8771824847322993.
[I 2026-05-27 19:41:31,336] Trial 4 finished with value: 0.8770744551069047 and parameters: {'C': 0.74617428116

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
77,77,0.878148,2026-05-27 19:41:42.619359,2026-05-27 19:41:42.763331,0 days 00:00:00.143972,0.003704,None,COMPLETE
89,89,0.878148,2026-05-27 19:41:44.456068,2026-05-27 19:41:44.590874,0 days 00:00:00.134806,0.001759,None,COMPLETE
67,67,0.878148,2026-05-27 19:41:41.065446,2026-05-27 19:41:41.209601,0 days 00:00:00.144155,0.002475,None,COMPLETE
43,43,0.878148,2026-05-27 19:41:37.376959,2026-05-27 19:41:37.531683,0 days 00:00:00.154724,0.003693,None,COMPLETE
94,94,0.878148,2026-05-27 19:41:45.213409,2026-05-27 19:41:45.362425,0 days 00:00:00.149016,0.003273,None,COMPLETE


#### Tryout best params

In [58]:
log_reg_recall_optimized_model = LogisticRegression(C=0.769946, class_weight='balanced', random_state=random_state)
log_reg_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = log_reg_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



In [59]:
log_reg_f1_optimized_model = LogisticRegression(C=0.315005, class_weight='balanced', random_state=random_state)
log_reg_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = log_reg_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



### Ridge

In [60]:
# We can tune alpha and class weight

#### Recall optimized

In [61]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_recall_optuna_results)

[I 2026-05-27 19:41:46,283] A new study created in memory with name: no-name-44805d05-dcdc-4f9e-9063-3d6162d2d914
[I 2026-05-27 19:41:46,430] Trial 0 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.6566939957934348, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:41:46,563] Trial 1 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.9977717390717432, 'class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:41:46,713] Trial 2 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.1540128095931137, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:41:46,862] Trial 3 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.5641405648316811, 'class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:41:47,016] Trial 4 finished with value: 0.8597323473365851 and parameters: {'alpha': 

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
0,0,0.859732,2026-05-27 19:41:46.284144,2026-05-27 19:41:46.430346,0 days 00:00:00.146202,0.656694,balanced,COMPLETE
1,1,0.859732,2026-05-27 19:41:46.430346,2026-05-27 19:41:46.563332,0 days 00:00:00.132986,0.997772,None,COMPLETE
2,2,0.859732,2026-05-27 19:41:46.564333,2026-05-27 19:41:46.713930,0 days 00:00:00.149597,0.154013,balanced,COMPLETE
3,3,0.859732,2026-05-27 19:41:46.713930,2026-05-27 19:41:46.862689,0 days 00:00:00.148759,0.564141,None,COMPLETE
4,4,0.859732,2026-05-27 19:41:46.863694,2026-05-27 19:41:47.016918,0 days 00:00:00.153224,0.097962,balanced,COMPLETE


#### Tryout best params

In [62]:
ridge_recall_optimized_model = RidgeClassifier(alpha=0.039558, class_weight='balanced', random_state=random_state)
ridge_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = ridge_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



#### F1 optimized

In [63]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_f1_optuna_results)

[I 2026-05-27 19:42:01,267] A new study created in memory with name: no-name-c7912c24-4a8f-41e6-8c2e-3ec3e6ef42e4
[I 2026-05-27 19:42:01,411] Trial 0 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.21178772925517617, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-05-27 19:42:01,558] Trial 1 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.6166011544438268, 'class_weight': None}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-05-27 19:42:01,706] Trial 2 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.20433576183630311, 'class_weight': None}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-05-27 19:42:01,848] Trial 3 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.45313503995566073, 'class_weight': None}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-05-27 19:42:02,007] Trial 4 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.4

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
0,0,0.876967,2026-05-27 19:42:01.267389,2026-05-27 19:42:01.411845,0 days 00:00:00.144456,0.211788,balanced,COMPLETE
1,1,0.876967,2026-05-27 19:42:01.412353,2026-05-27 19:42:01.557569,0 days 00:00:00.145216,0.616601,None,COMPLETE
2,2,0.876967,2026-05-27 19:42:01.559572,2026-05-27 19:42:01.706387,0 days 00:00:00.146815,0.204336,None,COMPLETE
3,3,0.876967,2026-05-27 19:42:01.707388,2026-05-27 19:42:01.848317,0 days 00:00:00.140929,0.453135,None,COMPLETE
4,4,0.876967,2026-05-27 19:42:01.849316,2026-05-27 19:42:02.007134,0 days 00:00:00.157818,0.497779,balanced,COMPLETE


#### Tryout best params

In [64]:
ridge_f1_optimized_model = RidgeClassifier(alpha=0.364826, class_weight='balanced', random_state=random_state)
ridge_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = ridge_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



### Lasso Logistic Regression

In [65]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [66]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15, n_jobs=-1)

lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_recall_optuna_results)

[I 2026-05-27 19:42:16,281] A new study created in memory with name: no-name-ca99bee2-9b7e-44d0-ae08-7f7a7f20a3fc
[I 2026-05-27 19:42:16,802] Trial 0 finished with value: 0.8597323473365851 and parameters: {'C': 0.8747370484101694, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:42:16,940] Trial 2 finished with value: 0.8597323473365851 and parameters: {'C': 0.5343430694797229, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:42:17,002] Trial 1 finished with value: 0.8597323473365851 and parameters: {'C': 0.6458865228276374, 'Class_weight': None}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:42:17,184] Trial 4 finished with value: 0.8597323473365851 and parameters: {'C': 0.1060621149544071, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:42:17,432] Trial 3 finished with value: 0.8597323473365851 and parameters: {'C': 0.900661283853

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
0,0,0.859732,2026-05-27 19:42:16.283854,2026-05-27 19:42:16.802589,0 days 00:00:00.518735,0.874737,balanced,COMPLETE
1,1,0.859732,2026-05-27 19:42:16.285854,2026-05-27 19:42:17.002652,0 days 00:00:00.716798,0.645887,None,COMPLETE
2,2,0.859732,2026-05-27 19:42:16.287934,2026-05-27 19:42:16.940454,0 days 00:00:00.652520,0.534343,balanced,COMPLETE
3,3,0.859732,2026-05-27 19:42:16.288933,2026-05-27 19:42:17.431605,0 days 00:00:01.142672,0.900661,balanced,COMPLETE
4,4,0.859732,2026-05-27 19:42:16.290933,2026-05-27 19:42:17.184872,0 days 00:00:00.893939,0.106062,balanced,COMPLETE


In [67]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_f1_optuna_results)

[I 2026-05-27 19:42:17,724] A new study created in memory with name: no-name-9740eeac-e10f-4a08-8792-21db5e43c16f
[I 2026-05-27 19:42:17,949] Trial 0 finished with value: 0.8771824847322993 and parameters: {'C': 0.5255259989557298, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-05-27 19:42:18,242] Trial 1 finished with value: 0.8771824847322993 and parameters: {'C': 0.9941381440000344, 'Class_weight': None}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-05-27 19:42:18,479] Trial 2 finished with value: 0.8771824847322993 and parameters: {'C': 0.5398796515169341, 'Class_weight': None}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-05-27 19:42:18,742] Trial 3 finished with value: 0.8771824847322993 and parameters: {'C': 0.8451907859901432, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8771824847322993.
[I 2026-05-27 19:42:19,033] Trial 4 finished with value: 0.8771824847322993 and parameters: {'C': 0.8423936978728491, 

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
0,0,0.877182,2026-05-27 19:42:17.726186,2026-05-27 19:42:17.949132,0 days 00:00:00.222946,0.525526,balanced,COMPLETE
1,1,0.877182,2026-05-27 19:42:17.950134,2026-05-27 19:42:18.242441,0 days 00:00:00.292307,0.994138,None,COMPLETE
2,2,0.877182,2026-05-27 19:42:18.242441,2026-05-27 19:42:18.479086,0 days 00:00:00.236645,0.539880,None,COMPLETE
3,3,0.877182,2026-05-27 19:42:18.480086,2026-05-27 19:42:18.742344,0 days 00:00:00.262258,0.845191,balanced,COMPLETE
4,4,0.877182,2026-05-27 19:42:18.743348,2026-05-27 19:42:19.033534,0 days 00:00:00.290186,0.842394,None,COMPLETE


#### Tryout best params

In [68]:
lasso_recall_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.248927, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = lasso_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [69]:
lasso_f1_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.496023, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = lasso_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## Conclusions

In [70]:
pd.concat(classification_reports, keys=classification_report_keys)

precision    recall  \
Baseline Logistic Regression            0              1.000000  0.007722   
                                        1              0.501697  1.000000   
                                        accuracy       0.503621  0.503621   
                                        macro avg      0.750848  0.503861   
                                        weighted avg   0.750969  0.503621   
Baseline Ridge Classifier               0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Baseline L1 (Lasso) Logistic Regression 0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Logistic Regression (Recall Optimized)  0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Logistic Regression (F1 Optimized)      0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Ridge Classifier (Recall Optimized)     0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Ridge Classifier (F1 Optimized)         0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Lasso Classifier (Recall Optimized)     0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Lasso Classifier (F1 Optimized)         0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   

                                                      f1-score      support  
Baseline Logistic Regression            0             0.015326  1036.000000  
                                        1             0.668173  1035.000000  
                                        accuracy      0.503621     0.503621  
                                        macro avg     0.341749  2071.000000  
                                        weighted avg  0.341592  2071.000000  
Baseline Ridge Classifier               0  

The lasso and ridge both overcome poor initial performance of the logistic regression model

The logistic reg performance could have been poor due to severe multicollinearity or imbalanced feature scaling

In [73]:
import numpy as np

pd.set_option('display.max_rows', None)

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Ridge_Coef': ridge_f1_optimized_model.coef_,
})

# Sort features by the absolute size of their coefficients
coef_df['Abs_Coefficient'] = np.abs(coef_df['Ridge_Coef'])
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)
coef_df

,Feature,Ridge_Coef,Abs_Coefficient
94,Payment Format_ACH,0.439854,0.439854
97,Payment Format_Cheque,-0.244206,0.244206
101,Account_Same,-0.235530,0.235530
98,Payment Format_Credit Card,-0.225024,0.225024
72,Receiving Currency_Saudi Riyal,0.203327,0.203327
87,Payment Currency_Saudi Riyal,-0.165397,0.165397
100,Payment Format_Wire,-0.123514,0.123514
96,Payment Format_Cash,-0.118390,0.118390
74,Receiving Currency_Swiss Franc,-0.108524,0.108524
89,Payment Currency_Swiss Franc,0.095514,0.095514


In [75]:
len(coef_df[coef_df['Abs_Coefficient'] == 0])

60

We can see 60 features coefs get shrunk to 0 by the Ridge model. These all consist of the _Mean and _Median computed features which aimed to give a summary of average transactions for a given currency. This shows these features did not add additional information to the model and can be safely dropped

In [77]:
coefficients = lasso_f1_optimized_model.coef_[0]

# 3. Create a DataFrame to map features to their coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coefficients
})

# 4. VIEW RETAINED FEATURES (Coefficient is NOT 0)
selected_features = coef_df[coef_df['Coefficient'] != 0]
print("Features Selected by Lasso:")
print(selected_features)

# 5. VIEW REJECTED FEATURES (Coefficient is exactly 0)
dropped_features = coef_df[coef_df['Coefficient'] == 0]
print("\nFeatures Ignored by Lasso:")
print(dropped_features['Feature'].tolist())

Features Selected by Lasso:
                                  Feature  Coefficient
0                         Amount Received     0.017789
1                             Amount Paid     0.017961
2                     Amount_Received_USD    -0.016400
3                         Amount_Paid_USD    -0.016484
64   Receiving Currency_Australian Dollar    -0.038141
65             Receiving Currency_Bitcoin    -0.011198
66         Receiving Currency_Brazil Real    -0.017142
67     Receiving Currency_Canadian Dollar    -0.053279
68                Receiving Currency_Euro     0.033876
69        Receiving Currency_Mexican Peso    -0.008931
70               Receiving Currency_Ruble    -0.007914
71               Receiving Currency_Rupee    -0.014150
72         Receiving Currency_Saudi Riyal     0.141448
73              Receiving Currency_Shekel    -0.074872
74         Receiving Currency_Swiss Franc    -0.049205
75            Receiving Currency_UK Pound    -0.023084
76           Receiving Currency_US Do

We can see Lasso also removed some features during feature selection. The features consist of the same summary statistic features which Ridge shrunk the coefficients of.